# PLN na USP: mapeamento de teses e dissertações

Notebook da análise: perfil da produção, classificação automática de **tarefas** e **métodos** de PLN, termos e tendências, modelagem de tópicos e validação com anotação manual.

Lê a base gerada pelo notebook `01_tratamento_coleta.ipynb`. Sem ela, roda com a amostra sintética `DATA/teses_sinteticas.csv` (trabalhos inventados).

## 1. Configuração

In [ ]:
from pathlib import Path

# Base tratada pelo notebook 01; sem ela, usa a amostra sintética para testes
ARQUIVO = Path('../DATA/teses_usp_pln.csv')
if not ARQUIVO.exists():
    ARQUIVO = Path('../DATA/teses_sinteticas.csv')
print('Usando', ARQUIVO.name)
# Planilha com a anotação manual de uma amostra (seção 8). Deixe como está se ainda não existir.
ARQUIVO_ANOTACAO = Path('../DATA/anotacao_manual.csv')

N_TOPICOS = 4          # com ~130 trabalhos, teste entre 4 e 8
MODELO_SPACY = 'pt_core_news_lg'

PASTA_RESULTADOS = Path('../DATA/resultados')
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from spacy.matcher import PhraseMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

try:
    nlp = spacy.load(MODELO_SPACY)
except OSError:
    # Modelo menor, caso o grande não esteja instalado
    nlp = spacy.load('pt_core_news_sm')
print('Modelo:', nlp.meta['name'])

## 2. Preparação dos dados

In [ ]:
df = pd.read_csv(ARQUIVO, dtype={'ano': 'Int64'})
n_inicial = len(df)

for col in ['titulo', 'palavras_chave', 'resumo_pt']:
    df[col] = df[col].fillna('').str.replace(r'\s+', ' ', regex=True).str.strip()

df = (df[df['resumo_pt'].str.len() > 0]
      .drop_duplicates(subset='url' if df['url'].notna().any() else 'titulo')
      .reset_index(drop=True))
df['nivel'] = df['nivel'].str.replace(r'^(Dissertação|Tese) de ', '', regex=True)

# Texto usado na classificação: título + palavras-chave + resumo
df['texto'] = df['titulo'] + '. ' + df['palavras_chave'].str.replace(';', ',') + '. ' + df['resumo_pt']

print(f'{n_inicial} trabalhos coletados → {len(df)} após a limpeza')
df[['ano', 'nivel', 'unidade', 'titulo']].head()

## 3. Perfil da produção

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pd.crosstab(df['ano'], df['nivel']).plot.bar(stacked=True, ax=ax[0], title='Trabalhos por ano e nível')
df['unidade'].value_counts().sort_values().plot.barh(ax=ax[1], title='Trabalhos por unidade')
plt.tight_layout()

## 4. Tarefas e métodos de PLN

Classificação por dicionário: cada trabalho recebe as tarefas e os métodos cujas expressões aparecem no título, nas palavras-chave ou no resumo. Os resumos em português costumam usar termos em inglês (*word embeddings*, *transformers*), por isso o dicionário mistura os dois idiomas.

O dicionário é um ponto de partida: revise-o lendo os trabalhos reais e valide-o com a anotação manual (seção 8).

In [ ]:
TAREFAS = {
    'sentimentos, opinião e posicionamento': ['análise de sentimento', 'análise de sentimentos', 'sentiment analysis',
                                              'mineração de opinião', 'opinion mining', 'polaridade',
                                              'posicionamento', 'posicionamentos', 'stance', 'retórica emocional'],
    'linguagem ofensiva e ironia': ['discurso de ódio', 'hate speech', 'ironia', 'sarcasmo'],
    'entidades e extração de informação': ['entidades nomeadas', 'reconhecimento de entidades', 'named entity',
                                           'extração de informação', 'extração de informações', 'information extraction'],
    'classificação de textos': ['classificação de textos', 'classificação de texto', 'classificação de documentos',
                                'classificação automática', 'text classification', 'categorização de textos'],
    'recuperação de informação': ['recuperação de informação', 'recuperação de informações', 'information retrieval',
                                  'pesquisa de jurisprudência', 'busca semântica'],
    'sumarização': ['sumarização', 'summarization', 'sumário automático'],
    'geração de texto': ['geração de texto', 'geração automática', 'geração de consultas', 'text generation'],
    'tradução automática': ['tradução automática', 'machine translation', 'auxílio à tradução'],
    'perguntas, diálogo e interfaces': ['perguntas e respostas', 'question answering', 'chatbot', 'agente conversacional',
                                        'agentes conversacionais', 'interfaces em linguagem natural',
                                        'interfaces baseadas em linguagem natural', 'sparql'],
    'semântica e desambiguação': ['desambiguação', 'desambiguação lexical', 'desambiguação léxica',
                                  'implicação textual', 'similaridade semântica'],
    'morfologia e sintaxe': ['análise sintática', 'analisador sintático', 'parser', 'parsing', 'treebank',
                             'etiquetagem', 'etiquetador', 'análise morfossintática', 'analisadores morfológicos',
                             'analisador morfológico', 'part-of-speech'],
    'avaliação e simplificação de textos': ['simplificação textual', 'simplificação de textos', 'legibilidade',
                                            'readability', 'compreensibilidade', 'qualidade de escrita',
                                            'avaliação automática de redações', 'correção automatizada'],
    'argumentação': ['argumentação', 'mineração de argumentos', 'argument mining'],
    'modelagem de tópicos': ['modelagem de tópicos', 'modelo de tópicos', 'topic modeling'],
    'mineração de textos (aplicações)': ['mineração de texto', 'mineração de textos', 'text mining'],
    'fala e fonologia': ['reconhecimento de fala', 'síntese de fala', 'speech recognition', 'text-to-speech',
                         'prosódia', 'fonologia'],
    'construção de recursos linguísticos': ['construção de um léxico', 'construção de léxico', 'construção de um corpus',
                                           'construção de corpus', 'corpora paralelos construídos', 'ontologia',
                                           'ontologias', 'terminologia', 'gestão terminológica', 'corpus eletrônico'],
}

METODOS = {  # em ordem cronológica aproximada
    '1. regras, lógica e léxicos': ['baseado em regras', 'baseada em regras', 'regras discursivas',
                                    'regras de simplificação', 'rule-based', 'gramática', 'gramáticas',
                                    'baseado em léxico', 'baseadas em léxico', 'autômatos', 'formalismos adaptativos',
                                    'lógica', 'reutilização de dicionários'],
    '2. métodos estatísticos e aprendizado de máquina clássico': [
        'aprendizado de máquina', 'machine learning', 'aprendizado computacional', 'aprendizagem computacional',
        'aprendizado automático', 'aprendizagem automático', 'aprendizagem automática', 'svm', 'naive bayes',
        'random forest', 'regressão logística', 'crf', 'cadeias de markov', 'maximização de entropia',
        'entropia máxima', 'processos estocásticos', 'clusterização', 'tf-idf'],
    '3. redes neurais e embeddings': ['rede neural', 'redes neurais', 'neural network', 'lstm', 'cnn', 'word2vec',
                                      'word embeddings', 'aprendizado profundo', 'aprendizagem profunda',
                                      'deep learning', 'representação vetorial', 'representações vetoriais'],
    '4. transformers e BERT': ['transformer', 'transformers', 'bert', 'bertimbau', 'roberta', 'ajuste fino'],
    '5. LLMs': ['llm', 'llms', 'grandes modelos de linguagem', 'grande modelo de linguagem',
                'large language model', 'gpt', 'chatgpt', 'llama', 'engenharia de prompt', 'prompts', 'zero-shot'],
}

def criar_matcher(dicionario):
    m = PhraseMatcher(nlp.vocab, attr='LOWER')
    for rotulo, termos in dicionario.items():
        m.add(rotulo, list(nlp.tokenizer.pipe(termos)))
    return m

m_tarefas, m_metodos = criar_matcher(TAREFAS), criar_matcher(METODOS)
docs = list(nlp.pipe(df['texto']))

def rotulos(matcher, doc):
    return sorted({nlp.vocab.strings[m_id] for m_id, _, _ in matcher(doc)})

df['tarefas'] = [rotulos(m_tarefas, d) for d in docs]
df['metodos'] = [rotulos(m_metodos, d) for d in docs]
sem_tarefa = df['tarefas'].str.len().eq(0).sum()
print(f'Trabalhos sem tarefa identificada: {sem_tarefa} de {len(df)} (revise o dicionário com eles)')
df[['ano', 'titulo', 'tarefas', 'metodos']].head(8)

In [ ]:
tarefas = df.explode('tarefas', ignore_index=True).dropna(subset=['tarefas'])
metodos = df.explode('metodos', ignore_index=True).dropna(subset=['metodos'])

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
tarefas['tarefas'].value_counts().sort_values().plot.barh(ax=ax[0], title='Tarefas de PLN (nº de trabalhos)')
pd.crosstab(metodos['ano'], metodos['metodos']).plot.bar(stacked=True, ax=ax[1],
                                                       title='Métodos por ano de defesa')
ax[1].legend(fontsize=8)
plt.tight_layout()

**PLN fora da Computação:** em quais unidades cada tarefa aparece.

In [ ]:
pd.crosstab(tarefas['unidade'], tarefas['tarefas'])

## 5. Termos e sintagmas nominais (spaCy)

In [ ]:
CLASSES = {'NOUN', 'PROPN', 'ADJ'}
# Palavras comuns a quase todo resumo, que não indicam tema
STOP_ACADEMICAS = {'trabalho', 'estudo', 'objetivo', 'resultado', 'análise', 'método', 'pesquisa',
                   'dado', 'abordagem', 'proposta', 'português', 'brasileiro', 'texto', 'linguagem'}

def lemas(doc):
    return [t.lemma_.lower() for t in doc
            if t.pos_ in CLASSES and not t.is_stop and t.is_alpha and len(t) > 2
            and t.lemma_.lower() not in STOP_ACADEMICAS]

def sintagmas(doc):
    saida = []
    for sn in doc.noun_chunks:
        tokens = [t for t in sn if not (t.is_stop and t.i == sn.start)]  # tira o artigo inicial
        if len(tokens) >= 2:
            saida.append(' '.join(t.text.lower() for t in tokens))
    return saida

docs_resumo = list(nlp.pipe(df['resumo_pt']))
df['lemas'] = [lemas(d) for d in docs_resumo]
df['sintagmas'] = [sintagmas(d) for d in docs_resumo]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
df.explode('lemas')['lemas'].value_counts().head(15).sort_values().plot.barh(ax=ax[0], title='Termos mais frequentes (lemas)')
df.explode('sintagmas')['sintagmas'].value_counts().head(15).sort_values().plot.barh(ax=ax[1], title='Sintagmas nominais mais frequentes')
plt.tight_layout()

**Termos emergentes:** comparação da frequência relativa entre a primeira e a segunda metade do período.

In [ ]:
meio = int(df['ano'].median())
por_periodo = (df.assign(periodo=df['ano'].le(meio).map({True: 'inicio', False: 'fim'}))
                 .explode('lemas')
                 .groupby(['lemas', 'periodo']).size().unstack(fill_value=0))
por_periodo = por_periodo.div(por_periodo.sum())     # frequência relativa em cada metade
por_periodo['variacao'] = por_periodo['fim'] - por_periodo['inicio']

emergentes = por_periodo.sort_values('variacao', ascending=False).head(10)
em_queda = por_periodo.sort_values('variacao').head(10)
print(f'Início: até {meio} | Fim: depois de {meio}')
pd.DataFrame({'emergente': emergentes.index, 'aumento': emergentes['variacao'].round(4).values,
              'em queda': em_queda.index, 'redução': em_queda['variacao'].round(4).values})

## 6. Modelagem de tópicos (TF-IDF + NMF)

Com a base real, teste alguns valores de `N_TOPICOS` e escolha o que gera tópicos mais interpretáveis. Compare os tópicos com as tarefas da seção 4: os tópicos podem revelar temas que o dicionário não prevê.

In [ ]:
vetorizador = TfidfVectorizer(max_df=0.9, min_df=1 if len(df) < 50 else 2, ngram_range=(1, 2))
X = vetorizador.fit_transform(df['lemas'].str.join(' '))

nmf = NMF(n_components=N_TOPICOS, random_state=42, init='nndsvda', max_iter=500)
W = nmf.fit_transform(X)
vocab = vetorizador.get_feature_names_out()

topicos = {f'T{i}': ', '.join(vocab[j] for j in comp.argsort()[::-1][:8])
           for i, comp in enumerate(nmf.components_)}
df['topico'] = [f'T{i}' for i in W.argmax(axis=1)]
pd.Series(topicos, name='termos principais').to_frame().join(df['topico'].value_counts().rename('n'))

## 7. Exportação dos resultados

In [ ]:
saida = df.drop(columns=['texto', 'resumo_pt', 'resumo_en', 'lemas', 'sintagmas'])
saida.to_csv(PASTA_RESULTADOS / 'trabalhos_classificados.csv', index=False)
pd.Series(topicos).to_csv(PASTA_RESULTADOS / 'topicos.csv', header=['termos'])
df[['url']].dropna().to_csv('../DATA/urls.csv', index=False)  # lista publicável para reconstruir a base
print('Resultados salvos em', PASTA_RESULTADOS.resolve())

## 8. Validação com anotação manual

Com cerca de 130 trabalhos, é viável **anotar manualmente** uma amostra (por exemplo, 40 trabalhos) e medir quanto a classificação automática acerta. Isso dá ao projeto um gabarito próprio.

1. A célula abaixo gera `DATA/anotacao_manual.csv` com uma amostra aleatória, se o arquivo ainda não existir.
2. Leia cada resumo e preencha as colunas `tarefa_manual` e `metodo_manual` com os rótulos dos dicionários (separe vários com `;`).
3. Rode a célula seguinte para calcular a precisão e a cobertura (*recall*) por rótulo.

In [ ]:
if not ARQUIVO_ANOTACAO.exists():
    amostra = df.sample(min(40, len(df)), random_state=42)[['titulo', 'resumo_pt']]
    amostra.assign(tarefa_manual='', metodo_manual='').to_csv(ARQUIVO_ANOTACAO, index_label='indice')
    print('Planilha de anotação criada em', ARQUIVO_ANOTACAO.resolve())
else:
    print('Planilha de anotação já existe:', ARQUIVO_ANOTACAO.resolve())

In [ ]:
def avaliar(coluna_auto, coluna_manual, rotulos_possiveis):
    anot = pd.read_csv(ARQUIVO_ANOTACAO, index_col='indice').dropna(subset=[coluna_manual])
    if anot.empty:
        return 'Nenhum trabalho anotado ainda.'
    manual = anot[coluna_manual].str.split(';').apply(lambda l: {r.strip() for r in l if r.strip()})
    auto = df.loc[anot.index, coluna_auto].apply(set)
    linhas = []
    for r in rotulos_possiveis:
        vp = sum(r in a and r in m for a, m in zip(auto, manual))
        fp = sum(r in a and r not in m for a, m in zip(auto, manual))
        fn = sum(r not in a and r in m for a, m in zip(auto, manual))
        if vp + fp + fn:
            linhas.append({'rótulo': r, 'precisão': vp / (vp + fp) if vp + fp else None,
                           'cobertura': vp / (vp + fn) if vp + fn else None, 'n_manual': vp + fn})
    return pd.DataFrame(linhas).round(2)

display(avaliar('tarefas', 'tarefa_manual', TAREFAS))
display(avaliar('metodos', 'metodo_manual', METODOS))